# 🧠 Distance Fusion MLP Training (Google Colab)

This notebook trains the **DistanceFusionMLP** model for **Track B** of the *Hack a Ton 2026* project.
It fuses geometric distance, relative depth score, and class ID to output a calibrated metric distance with uncertainty intervals.

## Environment Setup
Install requirements for Google Colab environment.

In [ ]:
# Install dependencies if not already present
!pip install torch numpy pandas onnx onnxruntime albumentations opencv-python

## 1. Imports and Config

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import pandas as pd
import os
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset and Model Definitions
We load the module implementations from `src/mlp_training/` directly or define them locally for convenience in Colab.

In [ ]:
# In Colab, you can upload the src/ directory or copy-paste classes here
# Local Definition of Model:
class DistanceFusionMLP(nn.Module):
    def __init__(self, input_dim: int = 3, hidden_dim: int = 64, dropout_rate: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 2)  # [distance, log_variance]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def gaussian_nll_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pred_dist = pred[:, 0]
    log_var = pred[:, 1]
    target = target.view(-1)
    precision = torch.exp(-log_var)
    loss = 0.5 * (log_var + ((target - pred_dist) ** 2) * precision)
    return torch.mean(loss)

## 3. Training Loop Configuration

In [ ]:
# Load synthetic or logged calibration data
# Ensure you have logged d_geometric_m, rel_depth_score, class_id, and true_distance_m
def generate_dummy_data(n_samples=500):
    np.random.seed(42)
    true_dist = np.random.uniform(1.0, 15.0, n_samples).astype(np.float32)
    # Add noise simulating sensor errors
    geom_dist = true_dist + np.random.normal(0, 0.15 * true_dist, n_samples).astype(np.float32)
    # Simulating a non-linear relative depth mapping
    rel_depth = (1.0 / (true_dist + 0.1)) + np.random.normal(0, 0.05, n_samples).astype(np.float32)
    rel_depth = np.clip(rel_depth, 0.0, 1.0)
    class_id = np.random.randint(0, 5, n_samples).astype(np.float32)
    
    df = pd.DataFrame({
        "d_geometric_m": geom_dist,
        "rel_depth_score": rel_depth,
        "class_id": class_id,
        "true_distance_m": true_dist
    })
    df.to_csv("data_log.csv", index=False)
    print("Dummy dataset created: data_log.csv")

generate_dummy_data()

In [ ]:
# Import and load dataset
from src.mlp_training.dataset import DistanceFusionDataset

dataset = DistanceFusionDataset("data_log.csv", is_training=True)
norm_params = dataset.get_norm_params()

# Save normalisation params config for edge deployment
dataset.save_norm_params("models/fusion_mlp_norm.json")
print(f"Saved normalization params: {norm_params}")

# Split into train/validation sets
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_set, val_set = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

## 4. Model Training

In [ ]:
model = DistanceFusionMLP(input_dim=3, hidden_dim=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

epochs = 100
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        pred = model(x_batch)
        loss = gaussian_nll_loss(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x_batch.size(0)
        
    scheduler.step()
    train_loss /= len(train_set)
    
    # Validation step
    model.eval()
    val_mae = 0.0
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(device), y_val.to(device)
            pred_val = model(x_val)[:, 0]
            val_mae += torch.sum(torch.abs(pred_val - y_val)).item()
            
    val_mae /= len(val_set)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val MAE: {val_mae:.3f}m")

## 5. Export to ONNX for Pi CPU Execution

In [ ]:
model.eval()
dummy_input = torch.randn(1, 3).to(device)
onnx_path = "models/fusion_mlp.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["features"],
    output_names=["prediction"],
    dynamic_axes={
        "features": {0: "batch"},
        "prediction": {0: "batch"}
    },
    opset_version=17
)
print(f"Exported model to ONNX: {onnx_path}")